# 01 — Exploratory Data Analysis (EDA)

Explore the Ahmedabad Metro crowd dataset to understand patterns, distributions, and correlations before building ML models.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

DATA_PATH = Path('../data/raw/ahmedabad_metro_bookings.csv')
df = pd.read_csv(DATA_PATH)
print(f'Dataset shape: {df.shape}')
df.head()

In [ ]:
# Dataset overview
print('\nColumn types:')
print(df.dtypes)
print('\nDescriptive stats:')
df.describe()

In [ ]:
# Missing values check
print('Missing values per column:')
print(df.isnull().sum())
print(f'\nTotal rows with missing values: {df.isnull().any(axis=1).sum()}')

In [ ]:
# Target distribution — crowd buckets
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bucket counts
bucket_order = ['Low', 'Medium', 'High']
colors = ['#22c55e', '#eab308', '#ef4444']
df['bucket'].value_counts().reindex(bucket_order).plot.bar(ax=axes[0], color=colors)
axes[0].set_title('Crowd Bucket Distribution')
axes[0].set_ylabel('Count')

# Actual crowd histogram
axes[1].hist(df['actual_crowd'], bins=30, color='#6366f1', edgecolor='white')
axes[1].set_title('Actual Crowd Distribution')
axes[1].set_xlabel('Crowd Count')
plt.tight_layout()
plt.show()

In [ ]:
# Hourly crowd pattern
hourly = df.groupby('hour')['actual_crowd'].agg(['mean', 'std'])

fig, ax = plt.subplots(figsize=(12, 5))
ax.fill_between(hourly.index, hourly['mean'] - hourly['std'], hourly['mean'] + hourly['std'], alpha=0.2, color='#6366f1')
ax.plot(hourly.index, hourly['mean'], color='#6366f1', linewidth=2, marker='o')
ax.axvspan(8, 10, alpha=0.1, color='red', label='Morning Peak')
ax.axvspan(17, 20, alpha=0.1, color='orange', label='Evening Peak')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Average Crowd')
ax.set_title('Crowd Pattern by Hour of Day')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap: Station x Hour
pivot = df.pivot_table(values='actual_crowd', index='station', columns='hour', aggfunc='mean')

fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(pivot, cmap='YlOrRd', annot=True, fmt='.0f', linewidths=0.5, ax=ax)
ax.set_title('Crowd Heatmap: Station x Hour')
plt.tight_layout()
plt.show()

In [ ]:
# Day of week pattern
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
daily = df.groupby('day_of_week')['actual_crowd'].mean()

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(7), daily.values, color=['#6366f1' if d < 5 else '#f59e0b' for d in range(7)])
ax.set_xticks(range(7))
ax.set_xticklabels(days)
ax.set_ylabel('Average Crowd')
ax.set_title('Crowd by Day of Week (weekdays vs weekends)')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix
numeric_cols = df.select_dtypes(include=[np.number]).columns

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', center=0, fmt='.2f', ax=ax)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Peak vs Off-peak comparison
df['is_peak_derived'] = df['hour'].apply(lambda h: 1 if (8 <= h <= 10) or (17 <= h <= 20) else 0)

fig, ax = plt.subplots(figsize=(8, 5))
df.boxplot(column='actual_crowd', by='is_peak_derived', ax=ax)
ax.set_xticklabels(['Off-Peak', 'Peak'])
ax.set_title('Crowd: Peak vs Off-Peak')
ax.set_xlabel('')
plt.suptitle('')
plt.tight_layout()
plt.show()

print(f"Peak avg crowd: {df[df['is_peak_derived']==1]['actual_crowd'].mean():.1f}")
print(f"Off-peak avg crowd: {df[df['is_peak_derived']==0]['actual_crowd'].mean():.1f}")